# **Imports**

In [3]:
import kagglehub
import shutil
import os
import torch
import numpy as np
from PIL import Image

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


# **Load Pretrained-ckpt**

In [ ]:
!mkdir -p /content/drive/MyDrive/VLU-Net-V2/checkpoints/pretrained_ckpt
!cp -r /content/drive/MyDrive/VLU-Net/checkpoints/pretrained_ckpt/* /content/drive/MyDrive/VLU-Net-V2/checkpoints/pretrained_ckpt/

# **Load Datasets**

## **Deblurring**

In [ ]:
# @title
target_dir = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro"
os.makedirs(os.path.dirname(target_dir), exist_ok=True)

print("Downloading dataset...")
cache_path = kagglehub.dataset_download("adwythdarsanr/gopro-image-deblurring-dataset")
print(f"Downloaded to temporary cache: {cache_path}")

gopro_source = os.path.join(cache_path, "Gopro")
if not os.path.exists(gopro_source):
    gopro_source = cache_path

print(f"Copying dataset contents directly to {target_dir}...")

if os.path.exists(target_dir):
    print(f"Removing pre-existing folder at {target_dir} to avoid duplication...")
    shutil.rmtree(target_dir)
os.makedirs(target_dir, exist_ok=True)

for item in os.listdir(gopro_source):
    source = os.path.join(gopro_source, item)
    destination = os.path.join(target_dir, item)

    if os.path.isdir(source):
        shutil.copytree(source, destination)
        print(f"Successfully copied folder: {item}")
    else:
        shutil.copy(source, destination)
        print(f"Successfully copied file: {item}")

print("\nDone! Your dataset contents are now copied safely to:", target_dir)

100%|██████████| 6.08G/6.08G [05:45<00:00, 18.9MB/s]

Extracting files...


Downloaded to temporary cache: /root/.cache/kagglehub/datasets/adwythdarsanr/gopro-image-deblurring-dataset/versions/1
Copying dataset contents directly to /content/VLU-Net-V2/datasets/deblurring_datasets/GoPro...
Successfully copied folder: test
Successfully copied folder: train

Done! Your dataset contents are now copied safely to: /content/VLU-Net-V2/datasets/deblurring_datasets/GoPro


## **Dehazing**

### **Training Dataset**

In [ ]:
base_target_dir = "/content/VLU-Net-V2/datasets/dehazing_datasets"
os.makedirs(base_target_dir, exist_ok=True)

# 1. Download to cache
print("Downloading OTS Training dataset...")
cache_path = kagglehub.dataset_download("brunobelloni/outdoor-training-set-ots-reside")

# 2. Define specific training destination
train_target_dir = os.path.join(base_target_dir, "OTS_outdoors")
os.makedirs(train_target_dir, exist_ok=True)

# 3. Move contents directly (bypassing the version folder)
print("Moving training files to Google Drive...")
for item in os.listdir(cache_path):
    source = os.path.join(cache_path, item)
    destination = os.path.join(train_target_dir, item)
    shutil.move(source, destination)

print(f"Training dataset ready at: {train_target_dir}")

100%|██████████| 11.1G/11.1G [10:10<00:00, 19.4MB/s]

Extracting files...


Moving training files to Google Drive...
Training dataset ready at: /content/VLU-Net-V2/datasets/dehazing_datasets/OTS_outdoors


### **Testing Dataset**

In [ ]:
base_target_dir = "/content/VLU-Net-V2/datasets/dehazing_datasets"
os.makedirs(base_target_dir, exist_ok=True)

# 1. Download to cache
print("Downloading SOTS Testing dataset...")
cache_path = kagglehub.dataset_download("balraj98/synthetic-objective-testing-set-sots-reside")
print(f"Dataset downloaded to cache: {cache_path}")

# 2. Define specific testing destination
test_target_dir = os.path.join(base_target_dir, "SOTS_outdoors")
os.makedirs(test_target_dir, exist_ok=True)

# 3. Move contents directly (bypassing the version folder)
print("Moving training files to Google Drive...")
source_outdoor_path = os.path.join(cache_path, "outdoor")
folders_to_move = ["clear", "hazy"]
for folder in folders_to_move:
    source_folder = os.path.join(source_outdoor_path, folder)
    destination_folder = os.path.join(test_target_dir, folder)

    if os.path.exists(source_folder):
        if os.path.exists(destination_folder):
            shutil.rmtree(destination_folder)

        shutil.move(source_folder, destination_folder)
        print(f"Moved {folder} folder to: {destination_folder}")
    else:
        print(f"Warning: Expected folder '{folder}' not found at {source_folder}")

print(f"Training dataset ready at: {train_target_dir}")

100%|██████████| 415M/415M [00:24<00:00, 17.5MB/s]

Extracting files...


Dataset downloaded to cache: /root/.cache/kagglehub/datasets/balraj98/synthetic-objective-testing-set-sots-reside/versions/1
Moving training files to Google Drive...
Moved clear folder to: /content/VLU-Net-V2/datasets/dehazing_datasets/SOTS_outdoors/clear
Moved hazy folder to: /content/VLU-Net-V2/datasets/dehazing_datasets/SOTS_outdoors/hazy
Training dataset ready at: /content/VLU-Net-V2/datasets/dehazing_datasets/OTS_outdoors


## **Delowlight**

In [ ]:
base_target_dir = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL"

print("Downloading LOL dataset...")
cache_path = kagglehub.dataset_download("soumikrakshit/lol-dataset")
print(f"Downloaded to temporary cache: {cache_path}")

source_root = os.path.join(cache_path, "lol_dataset")

if not os.path.exists(source_root):
    source_root = cache_path

folders_to_copy = ["eval15", "our485"]

print(f"Copying dataset components directly into {base_target_dir}...")

for folder in folders_to_copy:
    source_folder_path = os.path.join(source_root, folder)
    destination_folder_path = os.path.join(base_target_dir, folder)

    if os.path.exists(source_folder_path):
        if os.path.exists(destination_folder_path):
            print(f"Removing pre-existing destination folder: {destination_folder_path}")
            shutil.rmtree(destination_folder_path)

        os.makedirs(base_target_dir, exist_ok=True)

        shutil.copytree(source_folder_path, destination_folder_path)
        print(f"Successfully copied: {folder} -> {destination_folder_path}")
    else:
        print(f"Error: Expected folder '{folder}' was not found at {source_folder_path}")

print(f"\nDone! Your dataset folders are ready at:\n- {base_target_dir}/eval15\n- {base_target_dir}/our485")

100%|██████████| 331M/331M [00:19<00:00, 17.9MB/s]

Extracting files...


Downloaded to temporary cache: /root/.cache/kagglehub/datasets/soumikrakshit/lol-dataset/versions/1
Copying dataset components directly into /content/VLU-Net-V2/datasets/delowlight_datasets/LoL...
Successfully copied: eval15 -> /content/VLU-Net-V2/datasets/delowlight_datasets/LoL/eval15
Successfully copied: our485 -> /content/VLU-Net-V2/datasets/delowlight_datasets/LoL/our485

Done! Your dataset folders are ready at:
- /content/VLU-Net-V2/datasets/delowlight_datasets/LoL/eval15
- /content/VLU-Net-V2/datasets/delowlight_datasets/LoL/our485


## **Denoise**

### **Testing Dataset**
**CBSD68 & Urban100**

In [ ]:
cbsd68_target_dir = "/content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/clean"
urban100_target_dir = "/content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/clean"

print("Downloading Super Resolution Benchmarks dataset...")
cache_path = kagglehub.dataset_download("jesucristo/super-resolution-benchmarks")
print(f"Downloaded to temporary cache: {cache_path}")

cbsd68_source = os.path.join(cache_path, "CBSD68/CBSD68")
urban100_source = os.path.join(cache_path, "urban100/urban100")


if os.path.exists(cbsd68_source):
    print(f"Copying CBSD68 images directly to {cbsd68_target_dir}...")

    if os.path.exists(cbsd68_target_dir):
        shutil.rmtree(cbsd68_target_dir)
    os.makedirs(cbsd68_target_dir, exist_ok=True)

    for item in os.listdir(cbsd68_source):
        src_file = os.path.join(cbsd68_source, item)
        dst_file = os.path.join(cbsd68_target_dir, item)
        if os.path.isfile(src_file):
            shutil.copy(src_file, dst_file)

    print(f"CBSD68 successfully copied straight to: {cbsd68_target_dir}")
else:
    print(f"Error: Could not find CBSD68 source directory at {cbsd68_source}")


if os.path.exists(urban100_source):
    print(f"Copying Urban100 images directly to {urban100_target_dir}...")

    if os.path.exists(urban100_target_dir):
        shutil.rmtree(urban100_target_dir)
    os.makedirs(urban100_target_dir, exist_ok=True)

    for item in os.listdir(urban100_source):
        src_file = os.path.join(urban100_source, item)
        dst_file = os.path.join(urban100_target_dir, item)
        if os.path.isfile(src_file):
            shutil.copy(src_file, dst_file)

    print(f"Urban100 successfully copied straight to: {urban100_target_dir}")
else:
    print(f"Error: Could not find urban100 source directory at {urban100_source}")

print("\nDone! All files are resting directly inside their respective /clean folders.")

100%|██████████| 923M/923M [00:55<00:00, 17.6MB/s]

Extracting files...


Downloaded to temporary cache: /root/.cache/kagglehub/datasets/jesucristo/super-resolution-benchmarks/versions/8
Copying CBSD68 images directly to /content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/clean...
CBSD68 successfully copied straight to: /content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/clean
Copying Urban100 images directly to /content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/clean...
Urban100 successfully copied straight to: /content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/clean

Done! All files are resting directly inside their respective /clean folders.


### **Training Dataset**
**BSD400 & WED**

In [ ]:
bsd400_target_dir = "/content/VLU-Net-V2/datasets/denoising_datasets/BSD400/clean"
wed_target_dir = "/content/VLU-Net-V2/datasets/denoising_datasets/WED/clean"

os.makedirs(bsd400_target_dir, exist_ok=True)
os.makedirs(wed_target_dir, exist_ok=True)


print("Downloading WED dataset via kagglehub...")
wed_cache_path = kagglehub.dataset_download("hongshenelves/wed-dataset")
print(f"WED dataset downloaded to cache: {wed_cache_path}")

wed_source_dir = os.path.join(wed_cache_path, "WED")
if not os.path.exists(wed_source_dir):
    wed_source_dir = wed_cache_path

print(f"Copying WED dataset contents directly to {wed_target_dir}...")
if os.path.exists(wed_target_dir):
    shutil.rmtree(wed_target_dir)
os.makedirs(wed_target_dir, exist_ok=True)

for item in os.listdir(wed_source_dir):
    source = os.path.join(wed_source_dir, item)
    destination = os.path.join(wed_target_dir, item)

    if os.path.isdir(source):
        shutil.copytree(source, destination)
    else:
        shutil.copy(source, destination)

print(f"WED dataset successfully ready at: {wed_target_dir}\n")


print("Downloading BSD400 dataset from source repository...")

github_zip_url = "https://github.com/smartboy110/denoising-datasets/archive/832ae7be88063422bf1b63406e3028c5eae83ac7.zip"
zip_name = "denoising-datasets.zip"
extracted_dir = "denoising-datasets-832ae7be88063422bf1b63406e3028c5eae83ac7"

if os.path.exists(zip_name): os.remove(zip_name)
if os.path.exists(extracted_dir): shutil.rmtree(extracted_dir)

os.system(f"wget {github_zip_url} -O {zip_name}")
shutil.unpack_archive(zip_name, ".")

bsd400_source_path = os.path.join(extracted_dir, "BSD400")

if os.path.exists(bsd400_source_path):
    print(f"Filtering and copying BSD400 .png images to {bsd400_target_dir}...")

    if os.path.exists(bsd400_target_dir):
        shutil.rmtree(bsd400_target_dir)
    os.makedirs(bsd400_target_dir, exist_ok=True)

    copied_count = 0
    for item in os.listdir(bsd400_source_path):
        if item.lower().endswith('.png'):
            source_file = os.path.join(bsd400_source_path, item)
            destination_file = os.path.join(bsd400_target_dir, item)
            shutil.copy(source_file, destination_file)
            copied_count += 1

    print(f"BSD400 dataset successfully ready at: {bsd400_target_dir} ({copied_count} .png files copied)")
else:
    print("Error: Could not find the BSD400 directory inside the extracted archive.")

if os.path.exists(zip_name): os.remove(zip_name)
if os.path.exists(extracted_dir): shutil.rmtree(extracted_dir)

print("\nAll processes finished!")

100%|██████████| 2.12G/2.12G [02:02<00:00, 18.6MB/s]

Extracting files...


WED dataset downloaded to cache: /root/.cache/kagglehub/datasets/hongshenelves/wed-dataset/versions/1
Copying WED dataset contents directly to /content/VLU-Net-V2/datasets/denoising_datasets/WED/clean...
WED dataset successfully ready at: /content/VLU-Net-V2/datasets/denoising_datasets/WED/clean

Filtering and copying BSD400 .png images to /content/VLU-Net-V2/datasets/denoising_datasets/BSD400/clean...
BSD400 dataset successfully ready at: /content/VLU-Net-V2/datasets/denoising_datasets/BSD400/clean (400 .png files copied)

All processes finished!


## **Deraining**

Dataset link is in this [github](https://github.com/csdwren/RecDerain/blob/master/datasets/readme.txt) in the [baidu link](https://pan.baidu.com/s/1J0q6Mrno9aMCsaWZUtmbkg#list/path=%2F). If unable to access baidu drive, can get the dataset [here](https://pan.baidu.com/s/1J0q6Mrno9aMCsaWZUtmbkg#list/path=%2F)

In [ ]:
!mkdir -p /content/VLU-Net-V2/datasets/deraining_datasets
!cp -r /content/drive/MyDrive/VLU-Net-V2/datasets/deraining_datasets/* /content/VLU-Net-V2/datasets/deraining_datasets

## **Dataset Validation**

In [ ]:
paths_to_check = {
    # Deblurring
    "Deblur Test Blur": "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/test/blur",
    "Deblur Test Sharp": "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/test/sharp",
    "Deblur Train Blur": "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/train/blur",
    "Deblur Train Sharp": "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/train/sharp",

    # Dehazing
    "Dehaze OTS Clear": "/content/VLU-Net-V2/datasets/dehazing_datasets/OTS_outdoors/clear",
    "Dehaze OTS Hazy": "/content/VLU-Net-V2/datasets/dehazing_datasets/OTS_outdoors/hazy",
    "Dehaze SOTS GT": "/content/VLU-Net-V2/datasets/dehazing_datasets/SOTS_outdoors/clear",
    "Dehaze SOTS Hazy": "/content/VLU-Net-V2/datasets/dehazing_datasets/SOTS_outdoors/hazy",

    # Derain
    "Derain Rain100L Clear": "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/gt",
    "Derain Rain100L Rainy": "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/rainy",
    "Derain RainTrainL Clear": "/content/VLU-Net-V2/datasets/deraining_datasets/RainTrainL/gt",
    "Derain RainTrainL Rainy": "/content/VLU-Net-V2/datasets/deraining_datasets/RainTrainL/rainy",

    # Low-Light (LoL)
    "Low-Light Eval15 High": "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/eval15/high",
    "Low-Light Eval15 Low": "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/eval15/low",
    "Low-Light Our485 High": "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/our485/high",
    "Low-Light Our485 Low": "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/our485/low",

    # Denoising
    "Denoise BSD400 Clean": "/content/VLU-Net-V2/datasets/denoising_datasets/BSD400/clean",
    "Denoise Urban100 Clean": "/content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/clean",
    "Denoise CBSD68 Clean": "/content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/clean",
    "Denoise WED Clean": "/content/VLU-Net-V2/datasets/denoising_datasets/WED/clean"
}

print(f"{'Dataset Folder':<25} | {'File Count':<10}")
print("-" * 40)

# 3. Loop through and print the length of each folder
for name, path in paths_to_check.items():
    if os.path.exists(path):
        # Count only files, ignoring subdirectories if there are any
        num_files = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
        print(f"{name:<25} | {num_files:<10}")
    else:
        print(f"{name:<25} | PATH NOT FOUND ❌")

Dataset Folder            | File Count
----------------------------------------
Deblur Test Blur          | 1111      
Deblur Test Sharp         | 1111      
Deblur Train Blur         | 2103      
Deblur Train Sharp        | 2103      
Dehaze OTS Clear          | 2061      
Dehaze OTS Hazy           | 72135     
Dehaze SOTS GT            | 492       
Dehaze SOTS Hazy          | 500       
Derain Rain100L Clear     | 100       
Derain Rain100L Rainy     | 100       
Derain RainTrainL Clear   | 200       
Derain RainTrainL Rainy   | 600       
Low-Light Eval15 High     | 15        
Low-Light Eval15 Low      | 15        
Low-Light Our485 High     | 485       
Low-Light Our485 Low      | 485       
Denoise BSD400 Clean      | 400       
Denoise Urban100 Clean    | 100       
Denoise CBSD68 Clean      | 68        
Denoise WED Clean         | 4743      


# **Testing and Training Paths**

## **Denoising**

In [ ]:
def add_gaussian_noise(img, sigma):
    img = np.array(img).astype(np.float32) / 255.0
    noise = np.random.normal(0, sigma / 255.0, img.shape)
    noisy = np.clip(img + noise, 0, 1)
    return (noisy * 255).astype(np.uint8)


def generate_test_dataset(clean_dir, base_out, dataset_name):
    sigmas = [15, 25, 50]
    images = sorted(os.listdir(clean_dir))
    clean_folder_name = "clean"

    for sigma in sigmas:
        noisy_dir = os.path.join(base_out, f"noisy{sigma}")
        os.makedirs(noisy_dir, exist_ok=True)

        txt_path = os.path.join(base_out, f"{sigma}_test_paths.txt")

        with open(txt_path, "w") as f:
            for img_name in images:
                if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    continue

                clean_path = os.path.join(clean_dir, img_name)
                img = Image.open(clean_path).convert("RGB")
                noisy = add_gaussian_noise(img, sigma)

                noisy_path = os.path.join(noisy_dir, img_name)
                Image.fromarray(noisy).save(noisy_path)

                rel_noisy = f"./denoising_datasets/{dataset_name}/noisy{sigma}/{img_name}"
                rel_clean = f"./denoising_datasets/{dataset_name}/{clean_folder_name}/{img_name}"
                f.write(f"{rel_noisy},{rel_clean}\n")

    # Generate random sigma test set
    rand_txt = os.path.join(base_out, "rand_test_paths.txt")
    rand_dir = os.path.join(base_out, "noisy_rand")
    os.makedirs(rand_dir, exist_ok=True)

    with open(rand_txt, "w") as f:
        for img_name in images:
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                continue

            sigma = np.random.choice(sigmas)
            clean_path = os.path.join(clean_dir, img_name)
            img = Image.open(clean_path).convert("RGB")
            noisy = add_gaussian_noise(img, sigma)

            noisy_path = os.path.join(rand_dir, img_name)
            Image.fromarray(noisy).save(noisy_path)

            rel_noisy = f"./denoising_datasets/{dataset_name}/noisy_rand/{img_name}"
            rel_clean = f"./denoising_datasets/{dataset_name}/{clean_folder_name}/{img_name}"
            f.write(f"{rel_noisy},{rel_clean}\n")

    print(f"✅ Test dataset text paths and noisy directories built for {dataset_name}")

In [ ]:
base_denoising_dir = "/content/VLU-Net-V2/datasets/denoising_datasets"

generate_test_dataset(
    os.path.join(base_denoising_dir, "CBSD68/clean"),
    os.path.join(base_denoising_dir, "CBSD68"),
    "CBSD68"
)

generate_test_dataset(
    os.path.join(base_denoising_dir, "Urban100_HR/clean"),
    os.path.join(base_denoising_dir, "Urban100_HR"),
    "Urban100_HR"
)

✅ Test dataset text paths and noisy directories built for CBSD68
✅ Test dataset text paths and noisy directories built for Urban100_HR


In [ ]:
def generate_train_dataset(dataset_names, base_out):
    sigmas = [15, 25, 50]

    # Initialize the train path text files at the root level of denoising_datasets
    txt_files = {}
    for sigma in sigmas:
        txt_path = os.path.join(base_out, f"{sigma}_train_paths.txt")
        txt_files[sigma] = open(txt_path, "w")

    for dataset_name in dataset_names:
        possible_dirs = [
            os.path.join(base_out, dataset_name, "clean"),
            os.path.join(base_out, dataset_name, "original_png")
        ]

        clean_dir = None
        for d in possible_dirs:
            if os.path.exists(d):
                clean_dir = d
                break

        if clean_dir is None:
            print(f"⚠️ Warning: Clean directory not found in {possible_dirs}. Skipping {dataset_name}.")
            continue

        images = sorted(os.listdir(clean_dir))
        clean_folder = os.path.basename(os.path.normpath(clean_dir))

        for sigma in sigmas:
            noisy_dir = os.path.join(base_out, dataset_name, f"noisy{sigma}")
            os.makedirs(noisy_dir, exist_ok=True)

            for img_name in images:
                if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    continue

                clean_path = os.path.join(clean_dir, img_name)
                img = Image.open(clean_path).convert("RGB")
                noisy = add_gaussian_noise(img, sigma)

                noisy_path = os.path.join(noisy_dir, img_name)
                Image.fromarray(noisy).save(noisy_path)

                rel_noisy = f"./denoising_datasets/{dataset_name}/noisy{sigma}/{img_name}"
                rel_clean = f"./denoising_datasets/{dataset_name}/{clean_folder}/{img_name}"
                txt_files[sigma].write(f"{rel_noisy},{rel_clean}\n")

    for sigma in sigmas:
        txt_files[sigma].close()

    print("✅ Combined train dataset text paths and noisy directories built.")

In [ ]:
generate_train_dataset(["BSD400", "WED"], base_denoising_dir)

✅ Combined train dataset text paths and noisy directories built.


## **Delowlight**

In [ ]:
# =========================================================
# LOWLIGHT TRAIN
# =========================================================

low_dir = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/our485/low"
high_dir = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/our485/high"

relative_low = "./delowlight_datasets/LoL/our485/low"
relative_high = "./delowlight_datasets/LoL/our485/high"

output_txt = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/train_paths.txt"

files = sorted(os.listdir(low_dir))

with open(output_txt, "w") as f:
    for name in files:

        low_path = os.path.join(relative_low, name)
        high_path = os.path.join(relative_high, name)

        f.write(f"{low_path},{high_path}\n")


# =========================================================
# LOWLIGHT TEST
# =========================================================

low_dir = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/eval15/low"
high_dir = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/eval15/high"

relative_low = "./delowlight_datasets/LoL/eval15/low"
relative_high = "./delowlight_datasets/LoL/eval15/high"

output_txt = "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/test_paths.txt"

files = sorted(os.listdir(low_dir))

with open(output_txt, "w") as f:
    for name in files:

        low_path = os.path.join(relative_low, name)
        high_path = os.path.join(relative_high, name)

        f.write(f"{low_path},{high_path}\n")

## **Deblurring**

In [ ]:
# =========================================================
# DEBLURRING TRAIN
# =========================================================

low_dir = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/train/blur"
high_dir = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/train/sharp"

relative_low = "./deblurring_datasets/GoPro/train/blur"
relative_high = "./deblurring_datasets/GoPro/train/sharp"

output_txt = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/train_paths.txt"

files = sorted(os.listdir(low_dir))

with open(output_txt, "w") as f:
    for name in files:

        low_path = os.path.join(relative_low, name)
        high_path = os.path.join(relative_high, name)

        f.write(f"{low_path},{high_path}\n")


# =========================================================
# DEBLURRING TEST
# =========================================================

low_dir = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/test/blur"
high_dir = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/test/sharp"

relative_low = "./deblurring_datasets/GoPro/test/blur"
relative_high = "./deblurring_datasets/GoPro/test/sharp"

output_txt = "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/test_paths.txt"

files = sorted(os.listdir(low_dir))

with open(output_txt, "w") as f:
    for name in files:

        low_path = os.path.join(relative_low, name)
        high_path = os.path.join(relative_high, name)

        f.write(f"{low_path},{high_path}\n")

## **Dehazing**

In [ ]:
# =========================================================
# DEHAZING TRAIN (OTS)
# =========================================================
low_dir = "/content/VLU-Net-V2/datasets/dehazing_datasets/OTS_outdoors/hazy"
high_dir = "/content/VLU-Net-V2/datasets/dehazing_datasets/OTS_outdoors/clear"

relative_low = "./dehazing_datasets/OTS_outdoors/hazy"
relative_high = "./dehazing_datasets/OTS_outdoors/clear"

output_txt_train = "/content/VLU-Net-V2/datasets/dehazing_datasets/train_paths.txt"

clear_files = os.listdir(high_dir)
clear_map = {os.path.splitext(f)[0]: f for f in clear_files}

hazy_files = sorted(os.listdir(low_dir))

with open(output_txt_train, "w") as f:
    for name in hazy_files:
        base_id = name.split('_')[0]

        if base_id in clear_map:
            clear_name = clear_map[base_id]
            low_path = os.path.join(relative_low, name)
            high_path = os.path.join(relative_high, clear_name)

            f.write(f"{low_path},{high_path}\n")

# =========================================================
# DEHAZING TEST (SOTS)
# =========================================================
low_dir = "/content/VLU-Net-V2/datasets/dehazing_datasets/SOTS_outdoors/hazy"
high_dir = "/content/VLU-Net-V2/datasets/dehazing_datasets/SOTS_outdoors/clear"

relative_low = "./dehazing_datasets/SOTS_outdoors/hazy"
relative_high = "./dehazing_datasets/SOTS_outdoors/clear"

output_txt_test = "/content/VLU-Net-V2/datasets/dehazing_datasets/test_paths.txt"

clear_files_test = os.listdir(high_dir)
clear_map_test = {os.path.splitext(f)[0]: f for f in clear_files_test}

hazy_files_test = sorted(os.listdir(low_dir))

with open(output_txt_test, "w") as f:
    for name in hazy_files_test:
        base_id = name.split('_')[0]

        if base_id in clear_map_test:
            clear_name = clear_map_test[base_id]
            low_path = os.path.join(relative_low, name)
            high_path = os.path.join(relative_high, clear_name)

            f.write(f"{low_path},{high_path}\n")

## **Deraining**

In [7]:
import os

# =========================================================
# DERAINING TRAIN (RainTrainL)
# =========================================================
print("Generating path mapping for RainTrainL dataset...")

train_low_dir = "/content/VLU-Net-V2/datasets/deraining_datasets/RainTrainL/rainy"
train_high_dir = "/content/VLU-Net-V2/datasets/deraining_datasets/RainTrainL/gt"

train_relative_low = "./deraining_datasets/RainTrainL/rainy"
train_relative_high = "./deraining_datasets/RainTrainL/gt"

output_txt_train = "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/train_paths.txt"

train_pairs_count = 0

if os.path.exists(train_low_dir) and os.path.exists(train_high_dir):
    with open(output_txt_train, "w") as f:
        # Loop through all rainy images (rain-X.png, rainregion-X.png, rainstreak-X.png)
        for name in sorted(os.listdir(train_low_dir)):
            if not name.lower().endswith('.png'):
                continue

            # Extract the number from the file name
            # Example: "rainregion-1.png" -> split('-') -> ["rainregion", "1.png"] -> split('.') -> "1"
            try:
                img_num = name.split('-')[1].split('.')[0]
                corresponding_gt = f"norain-{img_num}.png"

                # Verify that the ground truth file actually exists
                if os.path.exists(os.path.join(train_high_dir, corresponding_gt)):
                    low_path = os.path.join(train_relative_low, name)
                    high_path = os.path.join(train_relative_high, corresponding_gt)

                    f.write(f"{low_path},{high_path}\n")
                    train_pairs_count += 1
            except IndexError:
                print(f"⚠️ Skipping unexpected file format: {name}")

    print(f"✅ RainTrainL file written! Total pairs mapped: {train_pairs_count}")
else:
    print("❌ Error: RainTrainL train directories not found.")


# =========================================================
# DERAINING TEST (Rain100L)
# =========================================================
print("\nGenerating path mapping for Rain100L dataset...")

test_low_dir = "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/rainy"
test_high_dir = "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/gt"

test_relative_low = "./deraining_datasets/Rain100L/rainy"
test_relative_high = "./deraining_datasets/Rain100L/gt"

output_txt_test = "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/test_paths.txt"

test_pairs_count = 0

if os.path.exists(test_low_dir) and os.path.exists(test_high_dir):
    with open(output_txt_test, "w") as f:
        for name in sorted(os.listdir(test_low_dir)):
            if not name.lower().endswith('.png'):
                continue

            try:
                # Example: "rain-001.png" -> "001"
                img_num = name.split('-')[1].split('.')[0]
                corresponding_gt = f"norain-{img_num}.png"

                if os.path.exists(os.path.join(test_high_dir, corresponding_gt)):
                    low_path = os.path.join(test_relative_low, name)
                    high_path = os.path.join(test_relative_high, corresponding_gt)

                    f.write(f"{low_path},{high_path}\n")
                    test_pairs_count += 1
            except IndexError:
                print(f"⚠️ Skipping unexpected file format: {name}")

    print(f"✅ Rain100L file written! Total pairs mapped: {test_pairs_count}")
else:
    print("❌ Error: Rain100L test directories not found.")

Generating path mapping for RainTrainL dataset...
✅ RainTrainL file written! Total pairs mapped: 600

Generating path mapping for Rain100L dataset...
✅ Rain100L file written! Total pairs mapped: 100


##**Validation**

In [ ]:
import os

# Updated dictionary with explicit LoL, GoPro, and Denoising Test structures
paths_to_check = {
    # --- Denoising Train (Combined) ---
    "Denoising Train (Sigma 15)": "/content/VLU-Net-V2/datasets/denoising_datasets/15_train_paths.txt",
    "Denoising Train (Sigma 25)": "/content/VLU-Net-V2/datasets/denoising_datasets/25_train_paths.txt",
    "Denoising Train (Sigma 50)": "/content/VLU-Net-V2/datasets/denoising_datasets/50_train_paths.txt",

    # --- Denoising Test (CBSD68) ---
    "Denoising Test (CBSD68 S15)": "/content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/15_test_paths.txt",
    "Denoising Test (CBSD68 S25)": "/content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/25_test_paths.txt",
    "Denoising Test (CBSD68 S50)": "/content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/50_test_paths.txt",
    "Denoising Test (CBSD68 Rand)": "/content/VLU-Net-V2/datasets/denoising_datasets/CBSD68/rand_test_paths.txt",

    # --- Denoising Test (Urban100_HR) ---
    "Denoising Test (Urban100 S15)": "/content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/15_test_paths.txt",
    "Denoising Test (Urban100 S25)": "/content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/25_test_paths.txt",
    "Denoising Test (Urban100 S50)": "/content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/50_test_paths.txt",
    "Denoising Test (Urban100 Rand)": "/content/VLU-Net-V2/datasets/denoising_datasets/Urban100_HR/rand_test_paths.txt",

    # --- Lowlight (LoL) ---
    "Lowlight Train (LoL)": "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/train_paths.txt",
    "Lowlight Test (LoL)":  "/content/VLU-Net-V2/datasets/delowlight_datasets/LoL/test_paths.txt",

    # --- Deblurring (GoPro) ---
    "Deblurring Train (GoPro)": "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/train_paths.txt",
    "Deblurring Test (GoPro)":  "/content/VLU-Net-V2/datasets/deblurring_datasets/GoPro/test_paths.txt",

    # --- Dehazing (OTS / SOTS) ---
    "Dehazing Train (OTS)": "/content/VLU-Net-V2/datasets/dehazing_datasets/train_paths.txt",
    "Dehazing Test (SOTS)": "/content/VLU-Net-V2/datasets/dehazing_datasets/test_paths.txt",

    # --- Deraining (Rain100L) ---
    "Deraining Train (RainTrainL)": "/content/VLU-Net-V2/datasets/deraining_datasets/RainTrainL/train_paths.txt",
    "Deraining Test (Rain100L)": "/content/VLU-Net-V2/datasets/deraining_datasets/Rain100L/test_paths.txt"
}

print("=== Checking Dataset Text Path Lengths ===\n")

for dataset_name, file_path in paths_to_check.items():
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            # Filter out empty or whitespace-only lines
            lines = [line for line in f.read().splitlines() if line.strip()]
            print(f"✅ {dataset_name:<30}: {len(lines):,} pairs")
    else:
        print(f"❌ {dataset_name:<30}: File not found at target path.")

=== Checking Dataset Text Path Lengths ===

✅ Denoising Train (Sigma 15)    : 5,143 pairs
✅ Denoising Train (Sigma 25)    : 5,143 pairs
✅ Denoising Train (Sigma 50)    : 5,143 pairs
✅ Denoising Test (CBSD68 S15)   : 68 pairs
✅ Denoising Test (CBSD68 S25)   : 68 pairs
✅ Denoising Test (CBSD68 S50)   : 68 pairs
✅ Denoising Test (CBSD68 Rand)  : 68 pairs
✅ Denoising Test (Urban100 S15) : 100 pairs
✅ Denoising Test (Urban100 S25) : 100 pairs
✅ Denoising Test (Urban100 S50) : 100 pairs
✅ Denoising Test (Urban100 Rand): 100 pairs
✅ Lowlight Train (LoL)          : 485 pairs
✅ Lowlight Test (LoL)           : 15 pairs
✅ Deblurring Train (GoPro)      : 2,103 pairs
✅ Deblurring Test (GoPro)       : 1,111 pairs
✅ Dehazing Train (OTS)          : 72,135 pairs
✅ Dehazing Test (SOTS)          : 500 pairs
✅ Deraining Train (RainTrainL)  : 600 pairs
✅ Deraining Test (Rain100L)     : 100 pairs


# **Zip Folder**

In [ ]:
%cd /content/VLU-Net-V2/datasets

/content/VLU-Net-V2/datasets


In [ ]:
!tar -cf deraining_datasets.tar deraining_datasets

In [ ]:
!tar -cf denoising_datasets.tar denoising_datasets

In [ ]:
!tar -cf dehazing_datasets.tar dehazing_datasets

In [ ]:
!tar -cf delowlight_datasets.tar delowlight_datasets

In [ ]:
!tar -cf deblurring_datasets.tar deblurring_datasets

## **Validate**

In [ ]:
!tar -tf dehazing_datasets.tar > /dev/null && echo "dehazing OK"

dehazing OK


In [ ]:
!tar -tf deblurring_datasets.tar > /dev/null && echo "deblurring OK"

deblurring OK


In [ ]:
!tar -tf delowlight_datasets.tar > /dev/null && echo "delowlight OK"

delowlight OK


In [ ]:
!tar -tf deraining_datasets.tar > /dev/null && echo "deraining OK"

deraining OK


In [ ]:
!tar -tf denoising_datasets.tar > /dev/null && echo "denoising OK"

denoising OK


In [ ]:
!ls -lh *.tar

-rw-r--r-- 1 root root 6.1G Jun 12 20:24 deblurring_datasets.tar
-rw-r--r-- 1 root root  12G Jun 12 20:24 dehazing_datasets.tar
-rw-r--r-- 1 root root 333M Jun 12 20:24 delowlight_datasets.tar
-rw-r--r-- 1 root root  13G Jun 12 20:22 denoising_datasets.tar
-rw-r--r-- 1 root root 166M Jun 12 20:21 deraining_datasets.tar


In [ ]:
!du -sh /content/VLU-Net-V2/datasets/*

6.1G	/content/VLU-Net-V2/datasets/deblurring_datasets
6.1G	/content/VLU-Net-V2/datasets/deblurring_datasets.tar
12G	/content/VLU-Net-V2/datasets/dehazing_datasets
12G	/content/VLU-Net-V2/datasets/dehazing_datasets.tar
335M	/content/VLU-Net-V2/datasets/delowlight_datasets
333M	/content/VLU-Net-V2/datasets/delowlight_datasets.tar
13G	/content/VLU-Net-V2/datasets/denoising_datasets
13G	/content/VLU-Net-V2/datasets/denoising_datasets.tar
167M	/content/VLU-Net-V2/datasets/deraining_datasets
166M	/content/VLU-Net-V2/datasets/deraining_datasets.tar


## **Upload Dataset Zip File**

In [ ]:
!mkdir -p /content/drive/MyDrive/VLU-Net-V2/datasets_cache

!cp /content/VLU-Net-V2/datasets/*.tar \
/content/drive/MyDrive/VLU-Net-V2/datasets_cache/

# **Load Dataset**

In [ ]:
%cd /content

/content


In [ ]:
!git clone https://github.com/xianggkl/VLU-Net.git
%cd VLU-Net

Cloning into 'VLU-Net'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 186 (delta 67), reused 180 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 11.84 MiB | 9.91 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/VLU-Net


In [ ]:
!mkdir -p /content/VLU-Net/datasets

In [ ]:
# copy + extract dehazing
!cp /content/drive/MyDrive/VLU-Net-V2/datasets_cache/dehazing_datasets.tar /content/
!tar -xf /content/dehazing_datasets.tar -C datasets

# copy + extract deraining
!cp /content/drive/MyDrive/VLU-Net-V2/datasets_cache/deraining_datasets.tar /content/
!tar -xf /content/deraining_datasets.tar -C datasets

# copy + extract delowlight
!cp /content/drive/MyDrive/VLU-Net-V2/datasets_cache/delowlight_datasets.tar /content/
!tar -xf /content/delowlight_datasets.tar -C datasets

# copy + extract deblurring
!cp /content/drive/MyDrive/VLU-Net-V2/datasets_cache/deblurring_datasets.tar /content/
!tar -xf /content/deblurring_datasets.tar -C datasets

# copy + extract denoising
!cp /content/drive/MyDrive/VLU-Net-V2/datasets_cache/denoising_datasets.tar /content/
!tar -xf /content/denoising_datasets.tar -C datasets

# **fix**

In [ ]:
import time

while True:
    time.sleep(600)
    print(".", end="", flush=True)

.............

KeyboardInterrupt: 

In [6]:
DATASET_ROOT = "/content/VLU-Net-V2/datasets"
CACHE_ROOT = "/content/drive/MyDrive/VLU-Net-V2/datasets_cache"

os.makedirs(DATASET_ROOT, exist_ok=True)

!tar -xf "{CACHE_ROOT}/deraining_datasets.tar" -C "{DATASET_ROOT}"

print("✓ All datasets extracted")

✓ All datasets extracted


In [8]:
!rm -rf /content/VLU-Net-V2/datasets/deraining_datasets/RainTrainL/train_paths.txt

In [9]:
%cd /content/VLU-Net-V2/datasets

/content/VLU-Net-V2/datasets


In [10]:
!tar -cf deraining_datasets.tar deraining_datasets

In [11]:
!tar -tf deraining_datasets.tar > /dev/null && echo "deraining OK"

deraining OK


In [12]:
!ls -lh *.tar

-rw-r--r-- 1 root root 166M Jun 13 23:32 deraining_datasets.tar


In [13]:
!du -sh /content/VLU-Net-V2/datasets/*

167M	/content/VLU-Net-V2/datasets/deraining_datasets
166M	/content/VLU-Net-V2/datasets/deraining_datasets.tar


In [14]:
!mkdir -p /content/drive/MyDrive/VLU-Net-V2/datasets_cache

!cp /content/VLU-Net-V2/datasets/*.tar \
/content/drive/MyDrive/VLU-Net-V2/datasets_cache/